# 06 Hybrid architecture: deterministic query tools + LLM router, scored on the 20-question benchmark

**Date:** 2026-10-22
**Author:** Xincheng (George) Ding
**Needs network** when `IATI_LIVE_EVAL=1` (Aura + the models listed in `.env`). Default is a dry run.

**Why.** Notebook 05 showed that a free Text2Cypher agent reaches 92 to 98% on the benchmark, with the remaining errors in
exactly the places a client cannot accept (a financial total). The SQL baseline is 100% because a human wrote each query.
This notebook makes that guarantee part of the product: every recurring question type becomes a **parameterised Cypher
template** (`fa26/src/tools/templates.py`, 16 tools incl. top funders, shared partners, funding over time, plus an
**entity resolver** in `resolver.py`). The LLM only chooses the tool and its arguments; it writes Cypher only when no tool
fits. A wrongly chosen tool can give the wrong answer; a chosen tool cannot give a wrong number.

**Three configurations on the same 20 questions, same scorer (`benchmark/score.py`):**

| config | who routes | who writes Cypher | expectation |
| --- | --- | --- | --- |
| A `free_cypher` (notebook 05 results, reused) | LLM | LLM | 92 to 98% |
| B `templates_manual` | a human map question to tool (ceiling, like the SQL baseline) | templates | 100% if the templates are right |
| C `hybrid_llm_router` | LLM picks tool + args, falls back to free Cypher | templates, LLM as fallback | between A and B; the product candidate |

**Outputs:** submissions in `fa26/data/processed/06_submission_<config>_<model>_<run>.json`, scores in
`fa26/benchmark/results/live_<run>_hybrid/`, long table `06_hybrid_eval_<run>.csv`.

In [ ]:
from pathlib import Path
import os, sys, re, json, time, subprocess, glob
from decimal import Decimal
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fa26" / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(FA26 / "benchmark")); sys.path.insert(0, str(FA26 / "src"))
from common import verify_snapshot, load_json, save_json
from text2cypher_eval import guard_query
from tools import TOOLS, run_tool, validate_args, tool_catalog, OrgResolver
SNAPSHOT_ID = verify_snapshot()["snapshot_id"]

import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv(FA26 / ".env", override=True)
DRY_RUN = os.environ.get("IATI_LIVE_EVAL", "0") != "1"
RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M")
PROCESSED = FA26 / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS = FA26 / "benchmark" / "results" / f"live_{RUN_ID}_hybrid"
QUESTIONS = load_json(FA26 / "benchmark" / "questions.json")
REASON_CODES = ["NEEDS_DATA_MISSING", "HISTORY_UNAVAILABLE", "FUNDING_DEFINITION_REQUIRED", "CURRENCY_BASIS_REQUIRED", "PERIOD_REQUIRED", "RAW_XML_UNAVAILABLE"]
print("run", RUN_ID, "| dry run:", DRY_RUN, "| tools:", len(TOOLS))

## 1. The tool layer (what Zhelin described as query tools v0.1, implemented against schema v0.1)

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"tool": n, "params": ", ".join(t["params"]) or "-", "description": t["description"][:110]} for n, t in TOOLS.items()])

In [ ]:
resolver = OrgResolver.from_csv(FA26 / "benchmark" / "snapshot" / "nodes_organisations.csv")
for q in ["UK government", "WFP", "ECHO", "DRC", "Oxfam Novib", "Multilateral"]:
    r = resolver.resolve(q); print(f"{q:<14} -> {r['kind']:<12} {r['refs'][:4]}   {r['note'][:70]}")
print("\nduplicate-looking identifier groups (merge list candidates):")
print(pd.DataFrame(resolver.duplicate_groups()).to_string(index=False))

## 2. Read-only executor with parameters

In [ ]:
if DRY_RUN:
    def run_cypher(q, params=None): return []
    print("dry run: no database")
else:
    from neo4j import GraphDatabase, Query, READ_ACCESS
    driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
    driver.verify_connectivity()
    DB = os.environ.get("NEO4J_DATABASE", "neo4j")
    def run_cypher(q, params=None):
        guard_query(q)
        with driver.session(database=DB, default_access_mode=READ_ACCESS) as s:
            return [dict(r) for r in s.run(Query(q, timeout=60), **(params or {}))]
    print("Neo4j connected; nodes:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"])

## 3. Shared formatting (same conventions as notebook 05)

In [ ]:
def fmt_value(v, kind):
    if kind == "integer": return str(int(round(float(v))))
    if kind == "decimal": return str(Decimal(str(v)).quantize(Decimal("0.01")))
    if kind == "date": return str(v)[:10]
    if isinstance(v, float) and not v.is_integer(): return str(Decimal(str(v)).quantize(Decimal("0.01")))
    if isinstance(v, (int, float)): return str(int(v))
    return str(v)

def rows_to_facts(rows, kind):
    return {str(r["key"]): {"value": fmt_value(r["value"], kind), "evidence": sorted({str(e) for e in (r.get("evidence") or []) if e})} for r in rows}

def make_answer(q, status, facts=None, reasons=None, explanation=None, latency_ms=None):
    a = {"id": q["id"], "status": status, "facts": facts or {} if status == "answer" else {},
         "reasons": reasons or [] if status != "answer" else [], "latency_ms": latency_ms}
    if explanation: a["explanation"] = explanation
    return a

## 4. Config B: templates with manual routing (the ceiling)

A human maps each benchmark question to a tool call. If this scores below 100%, the template is wrong, not the model.
The four limitation questions use fixed policy rules, as the SQL baseline does.

In [ ]:
ROUTE = {
    "Q01": ("activities_per_publisher", {}),
    "Q02": ("hotspot_countries_per_publisher", {}),
    "Q03": ("activities_per_publisher_by_country", {"country_code": "SO"}),
    "Q04": ("multi_publisher_participants", {"min_publishers": 2}),
    "Q05": ("participants_by_role", {}),
    "Q06": ("activities_by_match_basis", {}),
    "Q07": ("count_activities_by_sector", {"publisher": "wfp", "sector_key": "1:72040"}),
    "Q08": ("activities_per_publisher", {"start_from": "2020-01-01", "start_to": "2026-01-01"}),
    "Q09": ("latest_update_per_publisher", {}),
    "Q10": ("transaction_total", {"publisher": "wfp", "type_code": "1", "currency": "USD", "date_from": "2020-01-01", "date_to": "2021-01-01"}),
    "Q11": ("transaction_total", {"publisher": "oxfamgb", "type_code": "3", "currency": "GBP", "date_from": "2020-01-01", "date_to": "2021-01-01"}),
    "Q12": ("transaction_total", {"publisher": "oxfamus", "type_code": "4", "currency": "USD"}),
    "Q13": ("negative_transactions_per_publisher", {}),
    "Q14": ("budget_total", {"type_code": "1", "status_code": "2", "currency": "USD"}),
    "Q15": ("activities_per_sdg", {}),
    "Q16": ("activities_with_location_per_publisher", {}),
}
POLICY = {  # limitation questions: the snapshot cannot support these; same rules as the SQL baseline
    "Q17": ("abstain", ["NEEDS_DATA_MISSING"]), "Q18": ("abstain", ["HISTORY_UNAVAILABLE"]),
    "Q19": ("clarify", ["FUNDING_DEFINITION_REQUIRED", "CURRENCY_BASIS_REQUIRED", "PERIOD_REQUIRED"]), "Q20": ("abstain", ["RAW_XML_UNAVAILABLE"]),
}
records, submissions = [], []
answers = []
for q in QUESTIONS:
    t0 = time.time()
    if q["id"] in ROUTE:
        name, args = ROUTE[q["id"]]
        res = run_tool(run_cypher, name, args)
        ans = make_answer(q, "answer", rows_to_facts(res["rows"], q["value_type"]), latency_ms=round((time.time() - t0) * 1000, 1))
        records.append({"run": RUN_ID, "config": "templates_manual", "model": "none", "qid": q["id"], "status": "answer", "tool": name,
                        "args": json.dumps(args), "n_facts": len(ans["facts"]), "error": None, "latency_ms": ans["latency_ms"], "cypher": res["cypher"]})
    else:
        status, reasons = POLICY[q["id"]]
        ans = make_answer(q, status, reasons=reasons, explanation="policy rule", latency_ms=round((time.time() - t0) * 1000, 1))
        records.append({"run": RUN_ID, "config": "templates_manual", "model": "none", "qid": q["id"], "status": status, "tool": "policy",
                        "args": json.dumps(reasons), "n_facts": 0, "error": None, "latency_ms": ans["latency_ms"], "cypher": None})
    answers.append(ans)
    print(f"{q['id']} {ans['status']:<8} facts={len(ans['facts'])}  {records[-1]['tool']}")
p = PROCESSED / f"06_submission_templates_manual_{RUN_ID}.json"
save_json(p, {"run_name": f"templates_manual__neo4j_templates__{RUN_ID}", "snapshot_id": SNAPSHOT_ID, "answers": answers})
submissions.append(p); print("wrote", p.name)

## 5. Config C: hybrid agent (LLM router over tools, free Cypher as fallback) in LangGraph

`plan` asks the model for one JSON object: a status (answer / abstain / clarify), and for an answer either a tool name with
arguments or `free_cypher` with a query. `validate_args` rejects anything outside the tool's parameter schema, so the model
can only choose, never inject. Invalid arguments or a failed execution are fed back for up to 2 retries. The free-Cypher
fallback reuses the guard → execute → validate path from notebook 05.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage

SCHEMA_SHORT = '''Graph: (:Organisation {ref,name})-[:REPORTS|PARTICIPATES_IN {role}]->(:Activity {iati_identifier,publisher,match_basis,start_actual,start_planned,last_updated})
-[:TARGETS {is_hotspot}]->(:Country {code}); Activity-[:CLASSIFIED_AS]->(:Sector {sector_key}); Activity-[:LOCATED_AT]->(:Location);
Activity-[:CONTRIBUTES_TO]->(:SDG {sdg_key}); Activity-[:HAS_TRANSACTION]->(:Transaction {type '1'..'4', date, value_num, currency})-[:PROVIDED_BY|RECEIVED_BY]->(:Organisation);
Activity-[:HAS_BUDGET]->(:Budget {type,status,value,currency}). Every node/relationship has source_id. Publisher slugs: oxfamgb, oxfamus, wfp.
Transaction type codes: 1 incoming funds, 2 outgoing commitment, 3 disbursement, 4 expenditure. Dates are 'YYYY-MM-DD' strings.'''

SYSTEM = '''You route questions about one fixed snapshot of IATI humanitarian aid data to deterministic query tools.
Reply with ONE JSON object only:
{"status": "answer" | "abstain" | "clarify",
 "reasons": [],                       // abstain/clarify only, from ''' + json.dumps(REASON_CODES) + '''
 "explanation": "one sentence",
 "tool": "<tool name>" | "free_cypher" | null,
 "args": {...},                       // the tool's parameters, exactly as declared; omit optional ones you do not need
 "cypher": ""}                        // only when tool == "free_cypher": one read-only query returning key, value, evidence

Prefer a tool whenever one matches the question's contract exactly; use free_cypher only when no tool fits.
Abstain for needs-based rankings, historical versions or original XML fields; clarify when a funding question does not fix the
financial measure, currency basis and period. Country names must be converted to ISO-2 codes (Somalia = SO).

TOOLS:
''' + tool_catalog() + "\n\n" + SCHEMA_SHORT

class S(TypedDict, total=False):
    qid: str; question: str; contract: str; value_type: str
    raw: str; status: str; reasons: list; explanation: str; tool: Optional[str]; args: dict; cypher: str
    error: Optional[str]; retries: int; rows: list; facts: dict; model_calls: int

def parse_plan(text):
    m = re.search(r"\{.*\}", text, re.S); obj = json.loads(m.group(0)) if m else {}
    status = obj.get("status", "answer"); status = status if status in ("answer", "abstain", "clarify") else "answer"
    return status, [r for r in obj.get("reasons", []) if r in REASON_CODES], obj.get("explanation", ""), obj.get("tool"), obj.get("args") or {}, (obj.get("cypher") or "").strip().strip("`").rstrip(";")

def build_hybrid(llm, run_cypher):
    def plan(st):
        human = f"Question {st['qid']}: {st['question']}\nContract: {st['contract']}\nValue type: {st['value_type']}"
        if st.get("error"):
            human += f"\n\nYour previous attempt failed: {st['error']}\nPrevious choice: tool={st.get('tool')} args={json.dumps(st.get('args'))}\nReturn a corrected JSON object."
        out = llm.invoke([SystemMessage(content=SYSTEM), HumanMessage(content=human)])
        text = out.content if isinstance(out.content, str) else "\n".join(b.get("text", "") for b in out.content if isinstance(b, dict))
        try:
            status, reasons, expl, tool, args, cypher = parse_plan(text)
            err = None if status != "answer" or tool else "no tool chosen"
            return {"raw": text, "status": status, "reasons": reasons, "explanation": expl, "tool": tool, "args": args, "cypher": cypher,
                    "error": err, "rows": [], "model_calls": st.get("model_calls", 0) + 1}
        except Exception as e:
            return {"raw": text, "status": "answer", "tool": None, "error": f"unparseable JSON: {type(e).__name__}", "rows": [], "model_calls": st.get("model_calls", 0) + 1}

    def run_template(st):
        try:
            res = run_tool(run_cypher, st["tool"], st["args"])
            return {"rows": res["rows"], "cypher": res["cypher"], "args": res["params"], "error": None if res["rows"] or st["value_type"] == "decimal" else "tool returned no rows"}
        except Exception as e:
            return {"rows": [], "error": f"{type(e).__name__}: {str(e)[:300]}"}

    def run_free(st):
        try:
            guard_query(st["cypher"]); rows = run_cypher(st["cypher"])
        except Exception as e:
            return {"rows": [], "error": f"free_cypher failed {type(e).__name__}: {str(e)[:300]}"}
        for r in rows:
            if set(r) != {"key", "value", "evidence"}: return {"rows": [], "error": f"free_cypher must return key, value, evidence; got {sorted(r)}"}
        return {"rows": rows, "error": None if rows or st["value_type"] == "decimal" else "free_cypher returned no rows"}

    def format_(st):
        if st["status"] != "answer" or st.get("error"): return {"facts": {}}
        rows = st.get("rows") or [{"key": "total", "value": 0, "evidence": []}]
        return {"facts": rows_to_facts(rows, st["value_type"])}

    def after_plan(st):
        if st["status"] != "answer": return "format"
        if st.get("error"): return "retry" if st.get("retries", 0) < 2 else "format"
        return "free" if st["tool"] == "free_cypher" else "template"
    def after_exec(st): return ("retry" if st.get("retries", 0) < 2 else "format") if st.get("error") else "format"

    g = StateGraph(S)
    for n, f in [("plan", plan), ("template", run_template), ("free", run_free), ("format", format_), ("retry", lambda st: {"retries": st.get("retries", 0) + 1})]:
        g.add_node(n, f)
    g.set_entry_point("plan")
    g.add_conditional_edges("plan", after_plan, {"format": "format", "retry": "retry", "free": "free", "template": "template"})
    g.add_conditional_edges("template", after_exec, {"retry": "retry", "format": "format"})
    g.add_conditional_edges("free", after_exec, {"retry": "retry", "format": "format"})
    g.add_edge("retry", "plan"); g.add_edge("format", END)
    return g.compile()

In [ ]:
def env(k, default=None): return os.environ.get(k, default) or default
MODELS = []
if env("ANTHROPIC_API_KEY"):
    for m in (env("ANTHROPIC_EVAL_MODELS", "") or "").split(","):
        if m.strip(): MODELS.append({"name": f"anthropic/{m.strip()}", "provider": "anthropic", "model": m.strip()})
for prov in ("OPENAI", "GROQ", "NVIDIA", "ZAI", "DASHSCOPE", "DEEPSEEK", "MOONSHOT", "OPENROUTER"):
    if not env(f"{prov}_API_KEY"): continue
    url = env(f"{prov}_BASE_URL")
    if prov != "OPENAI" and not (url and url.startswith("https://")): continue
    for k, v in sorted(os.environ.items()):
        if (k == f"{prov}_MODEL" or k.startswith(f"{prov}_MODEL_")) and v.strip():
            MODELS.append({"name": f"{prov.lower()}/{v.strip()}", "provider": "openai_compat", "model": v.strip(), "key_env": f"{prov}_API_KEY", "base_url": url})
if DRY_RUN: MODELS = [{"name": "fake/router", "provider": "fake"}]
assert MODELS, "No models registered in fa26/.env"

def make_llm(spec):
    if spec["provider"] == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=spec["model"], temperature=0, max_tokens=1200, timeout=120)
    if spec["provider"] == "openai_compat":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=spec["model"], api_key=os.environ[spec["key_env"]], base_url=spec["base_url"], temperature=0, max_tokens=1200, timeout=180)
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    canned = [json.dumps({"status": "answer", "tool": "activities_per_publisher", "args": {}, "explanation": "dry"})] * 16 + \
             [json.dumps({"status": "abstain", "reasons": ["NEEDS_DATA_MISSING"], "tool": None, "explanation": "dry"})] * 4
    return FakeListChatModel(responses=canned * 3)
pd.DataFrame(MODELS)[["name", "provider"]]

In [ ]:
for spec in MODELS:
    agent = build_hybrid(make_llm(spec), run_cypher)
    answers = []
    for q in QUESTIONS:
        t0 = time.time()
        try:
            st = agent.invoke({"qid": q["id"], "question": q["question"], "contract": q["contract"], "value_type": q["value_type"], "retries": 0, "model_calls": 0})
        except Exception as e:
            st = {"status": "answer", "facts": {}, "reasons": [], "error": f"{type(e).__name__}: {str(e)[:200]}", "retries": 0}
        lat = round((time.time() - t0) * 1000, 1)
        ans = make_answer(q, st.get("status", "answer"), st.get("facts", {}), st.get("reasons", []), st.get("explanation"), lat)
        answers.append(ans)
        records.append({"run": RUN_ID, "config": "hybrid_llm_router", "model": spec["name"], "qid": q["id"], "status": ans["status"], "tool": st.get("tool"),
                        "args": json.dumps(st.get("args")), "n_facts": len(ans["facts"]), "error": st.get("error"), "retries": st.get("retries", 0),
                        "model_calls": st.get("model_calls", 0), "latency_ms": lat, "cypher": st.get("cypher")})
        print(f"{spec['name']:<26} {q['id']} {ans['status']:<8} tool={str(st.get('tool')):<40} facts={len(ans['facts']):<3} retries={st.get('retries',0)} {lat:>7.0f} ms {('ERR ' + st['error'][:60]) if st.get('error') else ''}")
    p = PROCESSED / f"06_submission_hybrid_{spec['name'].replace('/', '_')}_{RUN_ID}.json"
    save_json(p, {"run_name": f"hybrid_llm_router__{spec['name'].replace('/', '_')}__{RUN_ID}", "snapshot_id": SNAPSHOT_ID, "answers": answers})
    submissions.append(p); print("wrote", p.name)
results = pd.DataFrame(records); results.to_csv(PROCESSED / f"06_hybrid_eval_{RUN_ID}.csv", index=False)
results.groupby(["config", "model"]).agg(n=("qid", "count"), errors=("error", lambda s: s.notna().sum()), latency_ms=("latency_ms", "mean")).round(0)

## 6. Score all three configurations together (config A reuses the notebook 05 submissions)

In [ ]:
free_cypher = [Path(p) for p in sorted(glob.glob(str(PROCESSED / "05_submission_*.json"))) if "fake" not in p and "flash" not in p]
baseline = FA26 / "benchmark" / "runs" / "sqlite_baseline.json"
cmd = [sys.executable, str(FA26 / "benchmark" / "score.py"), *map(str, submissions), *map(str, free_cypher), str(baseline), "--output", str(RESULTS)]
proc = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
print(proc.stdout); print(proc.stderr[-2000:] if proc.returncode else "")
assert proc.returncode == 0, "scorer rejected a submission"

In [ ]:
details = load_json(RESULTS / "details.json")
rows = [{"run": r["summary"]["run_name"], "config": r["summary"]["run_name"].split("__")[0], "qid": q["id"], "score": round(q["score"], 2)}
        for r in details["runs"] for q in r["questions"]]
per_q = pd.DataFrame(rows)
summary = pd.DataFrame([r["summary"] for r in details["runs"]])[["run_name", "correctness_pct", "evidence_f1_pct", "behavior_pct", "overall_pct", "mean_latency_ms"]]
display(summary)
display(per_q.pivot(index="qid", columns="run", values="score"))

## 7. How to read it

- **templates_manual below 100%** means a template disagrees with the benchmark contract; fix the template (the Cypher is in
  `src/tools/templates.py`) and rerun this section only. This is the one place where a human still has to be right once.
- **hybrid above free_cypher** on the money questions (Q10 to Q14) is the product argument: the model picked the tool and
  the tool computed the number. Where hybrid is below, look at the `tool` and `args` columns in the CSV: the failure is a
  routing failure (wrong tool or wrong argument), which is fixable with examples, not a Cypher failure.
- **Evidence F1** for templates should be close to 100% because each template cites the same rows the benchmark reference
  uses (relationship rows plus the activities they join through). Shortfalls here are template design, not model behaviour.
- Next: wire these tools into a ReAct agent (notebook 07) so multi-step questions can chain them, and give the resolver's
  identifier groups to the pipeline team as the merge list.